## Implementation and Performance Evaluation of Categorical Naive Bayes Classifier

This notebook implements a classification pipeline using the Play Tennis dataset, covering data preprocessing, model training and evaluation, single-sample inference, and model comparison.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Google Sheets URL for the dataset
google_sheets_url = 'https://docs.google.com/spreadsheets/d/1mnoUgK8YxInzoDQ7P7iBM1HeZ8tcnUSvMU_H1OWndFA/edit?gid=0#gid=0'

# Convert to CSV export URL
csv_export_url = google_sheets_url.replace('/edit?gid=', '/export?format=csv&gid=')

# Load the dataset
df = pd.read_csv(csv_export_url)

print("Dataset loaded successfully:")
display(df.head())
print(f"\nDataset shape: {df.shape}")

Dataset loaded successfully:


,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes



Dataset shape: (50, 6)


### Data Preprocessing

1.  **Separate Features and Target:** The input features are 'Outlook', 'Temperature', 'Humidity', 'Wind', and the target variable is 'Play'.
2.  **Encode Categorical Variables:** All categorical features and the target label will be converted into numerical representations using `LabelEncoder`. This is particularly suitable for `CategoricalNB` which works well with integer-encoded categorical data.

In [5]:
# Separate input features (X) from the target variable (y)
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']].copy() # Explicitly create a copy
y = df['Play Tennis'] # Corrected column name

# Initialize LabelEncoders
le_features = {}
for column in X.columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])
    le_features[column] = le

le_target = LabelEncoder()
y = le_target.fit_transform(y)

print("Encoded Features (X):")
display(X.head())
print("\nEncoded Target (y):")
display(y[:5])

print("\nLabel Encoders for features:")
for col, le in le_features.items():
    print(f"{col}: {list(le.classes_)}")
print(f"Label Encoder for target: {list(le_target.classes_)}")

Encoded Features (X):


,Outlook,Temperature,Humidity,Wind
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1



Encoded Target (y):


array([0, 0, 1, 1, 1])


Label Encoders for features:
Outlook: ['Overcast', 'Rain', 'Sunny']
Temperature: ['Cool', 'Hot', 'Mild']
Humidity: ['High', 'Normal']
Wind: ['Strong', 'Weak']
Label Encoder for target: ['No', 'Yes']


### Dataset Partitioning

Dividing the dataset into training and testing subsets using an 80:20 train-test split ratio.

In [4]:
# Divide the dataset into training and testing subsets (80:20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape (features): {X_train.shape}")
print(f"Training set shape (target): {y_train.shape}")
print(f"Testing set shape (features): {X_test.shape}")
print(f"Testing set shape (target): {y_test.shape}")

Training set shape (features): (40, 4)
Training set shape (target): (40,)
Testing set shape (features): (10, 4)
Testing set shape (target): (10,)


### Naive Bayes Model Training & Evaluation

1.  **Train Categorical Naive Bayes (CategoricalNB) model.**
2.  **Predict class labels for the test dataset.**
3.  **Calculate and display overall Model Accuracy.**
4.  **Display Confusion Matrix and Classification Report (Precision, Recall, F1-Score).**

In [6]:
# Initialize and train the Categorical Naive Bayes model
cnb_model = CategoricalNB()
cnb_model.fit(X_train, y_train)

# Predict class labels for the test dataset
y_pred_cnb = cnb_model.predict(X_test)

# Calculate and display Model Accuracy
accuracy_cnb = accuracy_score(y_test, y_pred_cnb)
print(f"Categorical Naive Bayes Model Accuracy: {accuracy_cnb:.4f}\n")

# Display Confusion Matrix
conf_matrix_cnb = confusion_matrix(y_test, y_pred_cnb)
print("Confusion Matrix for Categorical Naive Bayes:")
print(conf_matrix_cnb)

# Display Classification Report
class_report_cnb = classification_report(y_test, y_pred_cnb, target_names=le_target.classes_)
print("\nClassification Report for Categorical Naive Bayes:")
print(class_report_cnb)

Categorical Naive Bayes Model Accuracy: 0.8000

Confusion Matrix for Categorical Naive Bayes:
[[1 1]
 [1 7]]

Classification Report for Categorical Naive Bayes:
              precision    recall  f1-score   support

          No       0.50      0.50      0.50         2
         Yes       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



### Single-Sample Inference

Predicting whether a person will play tennis under specific weather conditions and displaying both the predicted class label and the corresponding class probabilities.

In [7]:
# Define the specific weather conditions for prediction
single_sample_conditions = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

# Create a DataFrame for the single sample
single_sample_df = pd.DataFrame([single_sample_conditions])

# Encode the single sample's categorical features using the fitted LabelEncoders
single_sample_encoded = pd.DataFrame(index=single_sample_df.index)
for column, le in le_features.items():
    single_sample_encoded[column] = le.transform(single_sample_df[column])

# Predict the class label for the single sample
predicted_label_encoded = cnb_model.predict(single_sample_encoded)
predicted_label = le_target.inverse_transform(predicted_label_encoded)

# Predict the class probabilities for the single sample
predicted_proba = cnb_model.predict_proba(single_sample_encoded)

print(f"Specific weather conditions: {single_sample_conditions}")
print(f"Predicted Class Label: {predicted_label[0]}")

# Display probabilities for each class
print("Predicted Class Probabilities:")
for i, class_name in enumerate(le_target.classes_):
    print(f"  {class_name}: {predicted_proba[0][i]:.4f}")

Specific weather conditions: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class Label: No
Predicted Class Probabilities:
  No: 0.9256
  Yes: 0.0744


### Model Comparison

Training a Decision Tree Classifier, a Logistic Regression Classifier, and an SVM on the same training data. Comparing all models in terms of test accuracy, single-sample query prediction, and output class probabilities.

In [8]:
# --- Decision Tree Classifier ---
print("\n--- Decision Tree Classifier ---")
dtc_model = DecisionTreeClassifier(random_state=42)
dtc_model.fit(X_train, y_train)
y_pred_dtc = dtc_model.predict(X_test)
accuracy_dtc = accuracy_score(y_test, y_pred_dtc)
print(f"Decision Tree Model Accuracy: {accuracy_dtc:.4f}")

# Single-sample prediction for Decision Tree
predicted_label_dtc_encoded = dtc_model.predict(single_sample_encoded)
predicted_label_dtc = le_target.inverse_transform(predicted_label_dtc_encoded)[0]
predicted_proba_dtc = dtc_model.predict_proba(single_sample_encoded)[0]

print(f"Single-sample Predicted Class (Decision Tree): {predicted_label_dtc}")
print("Single-sample Predicted Probabilities (Decision Tree):")
for i, class_name in enumerate(le_target.classes_):
    print(f"  {class_name}: {predicted_proba_dtc[i]:.4f}")


# --- Logistic Regression Classifier ---
print("\n--- Logistic Regression Classifier ---")
lr_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' for small datasets and 'l1' penalty
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Model Accuracy: {accuracy_lr:.4f}")

# Single-sample prediction for Logistic Regression
predicted_label_lr_encoded = lr_model.predict(single_sample_encoded)
predicted_label_lr = le_target.inverse_transform(predicted_label_lr_encoded)[0]
predicted_proba_lr = lr_model.predict_proba(single_sample_encoded)[0]

print(f"Single-sample Predicted Class (Logistic Regression): {predicted_label_lr}")
print("Single-sample Predicted Probabilities (Logistic Regression):")
for i, class_name in enumerate(le_target.classes_):
    print(f"  {class_name}: {predicted_proba_lr[i]:.4f}")


# --- Support Vector Machine (SVM) Classifier ---
print("\n--- Support Vector Machine (SVM) Classifier ---")
# For SVM, use probability=True to get probabilities, but it can be computationally intensive
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM Model Accuracy: {accuracy_svm:.4f}")

# Single-sample prediction for SVM
predicted_label_svm_encoded = svm_model.predict(single_sample_encoded)
predicted_label_svm = le_target.inverse_transform(predicted_label_svm_encoded)[0]
predicted_proba_svm = svm_model.predict_proba(single_sample_encoded)[0]

print(f"Single-sample Predicted Class (SVM): {predicted_label_svm}")
print("Single-sample Predicted Probabilities (SVM):")
for i, class_name in enumerate(le_target.classes_):
    print(f"  {class_name}: {predicted_proba_svm[i]:.4f}")


# --- Comparison Table ---
print("\n--- Model Comparison ---")
comparison_data = {
    'Model': ['Categorical Naive Bayes', 'Decision Tree', 'Logistic Regression', 'SVM'],
    'Test Accuracy': [accuracy_cnb, accuracy_dtc, accuracy_lr, accuracy_svm],
    'Single-sample Prediction': [predicted_label[0], predicted_label_dtc, predicted_label_lr, predicted_label_svm],
    'Single-sample Proba (No)': [predicted_proba[0][0], predicted_proba_dtc[0], predicted_proba_lr[0], predicted_proba_svm[0]],
    'Single-sample Proba (Yes)': [predicted_proba[0][1], predicted_proba_dtc[1], predicted_proba_lr[1], predicted_proba_svm[1]]
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.round(4))




--- Decision Tree Classifier ---
Decision Tree Model Accuracy: 0.8000
Single-sample Predicted Class (Decision Tree): No
Single-sample Predicted Probabilities (Decision Tree):
  No: 1.0000
  Yes: 0.0000

--- Logistic Regression Classifier ---
Logistic Regression Model Accuracy: 0.4000
Single-sample Predicted Class (Logistic Regression): No
Single-sample Predicted Probabilities (Logistic Regression):
  No: 0.9466
  Yes: 0.0534

--- Support Vector Machine (SVM) Classifier ---
SVM Model Accuracy: 0.7000
Single-sample Predicted Class (SVM): No
Single-sample Predicted Probabilities (SVM):
  No: 0.9751
  Yes: 0.0249

--- Model Comparison ---


,Model,Test Accuracy,Single-sample Prediction,Single-sample Proba (No),Single-sample Proba (Yes)
0,Categorical Naive Bayes,0.8,No,0.9256,0.0744
1,Decision Tree,0.8,No,1.0000,0.0000
2,Logistic Regression,0.4,No,0.9466,0.0534
3,SVM,0.7,No,0.9751,0.0249
